# 🎬 BERT Sentiment Analysis — IMDb Fine-Tuning

This notebook fine-tunes `bert-base-uncased` on the [IMDb dataset](https://huggingface.co/datasets/stanfordnlp/imdb) for **binary sentiment classification** (positive / negative).

**Pipeline overview:**
1. Load & tokenize the IMDb dataset
2. Initialize `BertForSequenceClassification`
3. Fine-tune with the Hugging Face `Trainer` API
4. Evaluate with accuracy, F1, and a confusion matrix
5. Run inference on custom text

**Expected results (1 epoch, 2K train / 500 eval samples):** ~90% accuracy, ~0.90 F1

## 1. Setup & Imports

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from transformers import (
    BertForSequenceClassification,
    BertTokenizer,
    Trainer,
    TrainingArguments,
)

# Reproducibility
SEED = 42
torch.manual_seed(SEED)

# Config — adjust these to scale up training
MODEL_NAME     = "bert-base-uncased"
NUM_LABELS     = 2
TRAIN_SAMPLES  = 2_000   # Set to None to use the full dataset (~25K)
EVAL_SAMPLES   = 500     # Set to None to use the full test set
NUM_EPOCHS     = 1
BATCH_SIZE     = 8
LEARNING_RATE  = 2e-5
WEIGHT_DECAY   = 0.01
OUTPUT_DIR     = "./results"

print(f"Using device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 2. Load & Tokenize the Dataset

The [IMDb dataset](https://huggingface.co/datasets/stanfordnlp/imdb) contains 25K train and 25K test movie reviews, each labelled **0 (negative)** or **1 (positive)**.

In [ ]:
dataset = load_dataset("stanfordnlp/imdb")
print(dataset)

In [ ]:
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    """Tokenize a batch of examples with padding and truncation to BERT's max length."""
    return tokenizer(batch["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Optionally subsample for faster iteration
train_dataset = tokenized_datasets["train"].shuffle(seed=SEED)
eval_dataset  = tokenized_datasets["test"].shuffle(seed=SEED)

if TRAIN_SAMPLES:
    train_dataset = train_dataset.select(range(TRAIN_SAMPLES))
if EVAL_SAMPLES:
    eval_dataset = eval_dataset.select(range(EVAL_SAMPLES))

print(f"Train samples : {len(train_dataset)}")
print(f"Eval  samples : {len(eval_dataset)}")

## 3. Load the Model

`BertForSequenceClassification` adds a two-class linear head on top of BERT's `[CLS]` token.

> **Expected warnings:** The `classifier` head weights (`classifier.weight`, `classifier.bias`) are *missing* from the pre-trained checkpoint — they are randomly initialised and learned during fine-tuning. The `cls.predictions.*` and `cls.seq_relationship.*` keys are pre-training-only weights that are safely ignored.

In [ ]:
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
print(f"Model parameters: {model.num_parameters():,}")

## 4. Training

In [ ]:
def compute_metrics(eval_pred):
    """Compute accuracy and macro F1 from Trainer evaluation predictions."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="macro"),
    }

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    logging_steps=50,
    report_to="none",   # Set to 'wandb' or 'tensorboard' to enable experiment tracking
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

## 5. Evaluation

In [ ]:
results = trainer.evaluate()

print("\n── Evaluation Results ──────────────────")
print(f"  Loss     : {results['eval_loss']:.4f}")
print(f"  Accuracy : {results['eval_accuracy']:.4f}")
print(f"  F1       : {results['eval_f1']:.4f}")

In [ ]:
# Confusion matrix
predictions_output = trainer.predict(eval_dataset)
preds  = np.argmax(predictions_output.predictions, axis=-1)
labels = predictions_output.label_ids

cm = confusion_matrix(labels, preds)
label_names = ["Negative", "Positive"]

print("\n── Confusion Matrix ────────────────────")
print(f"{'':>12}  Pred Neg  Pred Pos")
for i, row_name in enumerate(label_names):
    print(f"  {row_name:>10}  {cm[i][0]:>8}  {cm[i][1]:>8}")

## 6. Inference on Custom Text

Run the fine-tuned model on any sentences you like.

In [ ]:
def predict_sentiment(texts: list[str]) -> list[str]:
    """
    Predict sentiment for a list of raw text strings.

    Args:
        texts: List of review strings.

    Returns:
        List of 'Positive' / 'Negative' labels.
    """
    model.eval()
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    preds = torch.argmax(logits, dim=1).cpu().tolist()
    return ["Positive" if p == 1 else "Negative" for p in preds]


# ── Demo ────────────────────────────────────────────────────────────────────
sample_texts = [
    "old and good",
    "I LOVE THIS MOVIE",
    "Absolutely dreadful. A waste of two hours.",
    "One of the best films I've seen in years — brilliant performances.",
    "It was okay, nothing special.",
]

sentiments = predict_sentiment(sample_texts)

print("\n── Predictions ─────────────────────────")
for text, sentiment in zip(sample_texts, sentiments):
    print(f"  [{sentiment:>8}]  {text}")